# CogniStream — Moondream2 Fine-tuning

Fine-tunes moondream2's text model on NVIDIA-distilled video captions.
Vision encoder is frozen (official recommendation).

**Steps:**
1. Upload `distillation_dataset.jsonl` and keyframe images
2. Train (2 epochs, ~5-10 min on Colab GPU)
3. Download the fine-tuned model
4. Convert to GGUF for Ollama

In [ ]:
# 1. Install dependencies
!pip install -q transformers torch pillow einops accelerate

In [ ]:
# 2. Upload your data
# Option A: Upload from Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy your data folder (upload distillation_dataset.jsonl + frames to Drive first)
# !cp /content/drive/MyDrive/cognistream_finetune/distillation_dataset.jsonl /content/
# !cp -r /content/drive/MyDrive/cognistream_finetune/frames/ /content/frames/

# Option B: Upload directly via Colab file upload
from google.colab import files
print("Upload distillation_dataset.jsonl:")
uploaded = files.upload()

In [ ]:
# 3. Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# 4. Load dataset
import json
from pathlib import Path

DATASET_PATH = "distillation_dataset.jsonl"

items = []
with open(DATASET_PATH) as f:
    for line in f:
        items.append(json.loads(line))

print(f"Loaded {len(items)} training examples")

# Check how many have valid image paths
# (paths will be different on Colab — we only use the text for training)
print(f"Sample response: {items[0]['response'][:200]}")

In [ ]:
# 5. Load moondream2
from transformers import AutoModelForCausalLM, AutoTokenizer

MD_REVISION = "2024-08-26"
DEVICE = "cuda"
DTYPE = torch.float16

print(f"Loading moondream2 (revision={MD_REVISION})...")
tokenizer = AutoTokenizer.from_pretrained(
    "vikhyatk/moondream2", revision=MD_REVISION, trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2", revision=MD_REVISION, trust_remote_code=True,
    dtype=DTYPE, device_map={"":DEVICE},
)
print("Model loaded.")

In [ ]:
# 6. Freeze vision encoder, train only text model
import torch.nn.functional as F

vision_frozen = 0
text_trainable = 0
for name, param in model.named_parameters():
    if "vision" in name.lower() or "visual" in name.lower() or "enc" in name.lower():
        param.requires_grad = False
        vision_frozen += param.numel()
    else:
        param.requires_grad = True
        text_trainable += param.numel()

print(f"Vision encoder: {vision_frozen/1e6:.0f}M params (FROZEN)")
print(f"Text model: {text_trainable/1e6:.0f}M params (TRAINABLE)")

# Get internal text model components
inner = model.model if hasattr(model, 'model') else model
wte = inner.text.wte if hasattr(inner, 'text') else None
text_model = inner.text

# Get moondream's own tokenizer
md_tokenizer = inner.tokenizer if hasattr(inner, 'tokenizer') else None

print(f"Text embedding: {wte.shape if wte is not None else 'NOT FOUND'}")
print(f"MD tokenizer: {'Found' if md_tokenizer else 'Using HF tokenizer'}")

In [ ]:
# 7. Training
import random
import time

EPOCHS = 2
GRAD_ACCUM = 8
LR = 3e-5

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)

model.train()
total_steps = (len(items) * EPOCHS) // GRAD_ACCUM
step = 0
t_start = time.monotonic()

for epoch in range(EPOCHS):
    random.shuffle(items)
    epoch_loss = 0.0
    batch_loss = 0.0
    valid = 0

    for i, item in enumerate(items):
        try:
            answer = item["response"]

            # Tokenize
            if md_tokenizer:
                token_ids = md_tokenizer.encode(answer).ids
            else:
                token_ids = tokenizer.encode(answer, add_special_tokens=False)

            if len(token_ids) < 5:
                continue

            token_ids = token_ids[:512]
            input_ids = torch.tensor([token_ids], device=DEVICE)

            # Embed tokens
            embeds = F.embedding(input_ids, wte)

            # Forward through text model blocks
            hidden = embeds
            for block in text_model.blocks:
                ln_weight = block.ln.weight
                ln_bias = block.ln.bias if hasattr(block.ln, 'bias') else None
                normed = F.layer_norm(hidden, (hidden.size(-1),), ln_weight, ln_bias)

                # Attention
                qkv = block.attn.qkv(normed)
                d = hidden.size(-1)
                q, k, v = qkv.chunk(3, dim=-1) if qkv.size(-1) == d * 3 else qkv.split([d, d//4*2, d//4*2], dim=-1)

                bsz, seq_len = hidden.shape[:2]
                head_dim = 64
                n_heads = d // head_dim
                n_kv = k.size(-1) // head_dim

                q_h = q[..., :d].view(bsz, seq_len, n_heads, head_dim).transpose(1, 2)
                k_h = k.view(bsz, seq_len, n_kv, head_dim).transpose(1, 2)
                v_h = v.view(bsz, seq_len, n_kv, head_dim).transpose(1, 2)

                causal = torch.triu(torch.ones(seq_len, seq_len, device=DEVICE), diagonal=1).bool()
                attn_out = F.scaled_dot_product_attention(q_h, k_h, v_h, attn_mask=~causal.unsqueeze(0).unsqueeze(0))
                attn_out = attn_out.transpose(1, 2).reshape(bsz, seq_len, d)
                attn_out = block.attn.proj(attn_out)

                # MLP
                mlp_out = block.mlp.fc2(F.gelu(block.mlp.fc1(normed)))
                hidden = hidden + attn_out + mlp_out

            # LM head
            hidden = F.layer_norm(hidden, (hidden.size(-1),),
                                  text_model.post_ln.weight,
                                  text_model.post_ln.bias if hasattr(text_model.post_ln, 'bias') else None)
            logits = text_model.lm_head(hidden)

            # Loss
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                input_ids[:, 1:].contiguous().view(-1),
            )

            loss = loss / GRAD_ACCUM
            loss.backward()
            batch_loss += loss.item()
            valid += 1

            if valid % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
                optimizer.zero_grad()
                step += 1

                if step % 5 == 0:
                    elapsed = time.monotonic() - t_start
                    print(f"  Epoch {epoch+1} | Step {step}/{total_steps} | Loss: {batch_loss:.4f} | {elapsed/60:.1f} min")
                epoch_loss += batch_loss
                batch_loss = 0.0

        except Exception as exc:
            if i < 3:
                print(f"  Sample {i} error: {exc}")
            continue

    n_steps = max(1, valid // GRAD_ACCUM)
    print(f"Epoch {epoch+1} complete | Avg loss: {epoch_loss/n_steps:.4f} | Valid: {valid}/{len(items)}")

elapsed = time.monotonic() - t_start
print(f"\nTraining complete: {step} steps in {elapsed/60:.1f} min")

In [ ]:
# 8. Save fine-tuned model
OUTPUT_DIR = "/content/cognistream-moondream"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

# Check size
import os
total = sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fns in os.walk(OUTPUT_DIR) for f in fns)
print(f"Model size: {total/1e9:.1f} GB")

In [ ]:
# 9. Test the fine-tuned model
model.eval()

test_prompt = """Analyze this image. Respond with EXACTLY this format:
SCENE: [2-3 sentence description of setting and atmosphere]
OBJECTS: [comma-separated list of visible objects]
ACTIVITY: [one sentence describing what is happening]
ANOMALY: [anything unusual, or 'none']"""

# Generate without image (text-only test)
if md_tokenizer:
    ids = md_tokenizer.encode(test_prompt).ids[:256]
else:
    ids = tokenizer.encode(test_prompt, add_special_tokens=False)[:256]

input_ids = torch.tensor([ids], device=DEVICE)
with torch.no_grad():
    embeds = F.embedding(input_ids, wte)
    hidden = embeds
    for block in text_model.blocks:
        normed = F.layer_norm(hidden, (hidden.size(-1),), block.ln.weight,
                              block.ln.bias if hasattr(block.ln, 'bias') else None)
        qkv = block.attn.qkv(normed)
        d = hidden.size(-1)
        q, k, v = qkv.chunk(3, dim=-1) if qkv.size(-1) == d * 3 else qkv.split([d, d//4*2, d//4*2], dim=-1)
        bsz, seq_len = hidden.shape[:2]
        head_dim = 64
        n_heads = d // head_dim
        n_kv = k.size(-1) // head_dim
        q_h = q[..., :d].view(bsz, seq_len, n_heads, head_dim).transpose(1, 2)
        k_h = k.view(bsz, seq_len, n_kv, head_dim).transpose(1, 2)
        v_h = v.view(bsz, seq_len, n_kv, head_dim).transpose(1, 2)
        causal = torch.triu(torch.ones(seq_len, seq_len, device=DEVICE), diagonal=1).bool()
        attn_out = F.scaled_dot_product_attention(q_h, k_h, v_h, attn_mask=~causal.unsqueeze(0).unsqueeze(0))
        attn_out = attn_out.transpose(1, 2).reshape(bsz, seq_len, d)
        attn_out = block.attn.proj(attn_out)
        mlp_out = block.mlp.fc2(F.gelu(block.mlp.fc1(normed)))
        hidden = hidden + attn_out + mlp_out

    hidden = F.layer_norm(hidden, (hidden.size(-1),), text_model.post_ln.weight,
                          text_model.post_ln.bias if hasattr(text_model.post_ln, 'bias') else None)
    logits = text_model.lm_head(hidden)
    next_token = logits[:, -1, :].argmax(dim=-1)

if md_tokenizer:
    print(f"Next token prediction: {md_tokenizer.decode([next_token.item()])}")
else:
    print(f"Next token prediction: {tokenizer.decode(next_token)}")
print("Model is working!")

In [ ]:
# 10. Download the fine-tuned model
# Option A: Zip and download directly
!cd /content && zip -r cognistream-moondream.zip cognistream-moondream/
files.download("/content/cognistream-moondream.zip")

# Option B: Save to Google Drive
# !cp -r /content/cognistream-moondream /content/drive/MyDrive/cognistream_finetune/

## After Download

1. Extract `cognistream-moondream.zip` to `data/finetune/cognistream-moondream/`
2. Convert to GGUF:
   ```bash
   python scripts/finetune/export_ollama.py
   ```
3. Set in `.env`:
   ```
   OLLAMA_MODEL=cognistream-vlm
   ```